# 📋 Notebook 5 — Model Evaluation

**Project:** Customer Churn Prediction  
**Objective:** In-depth evaluation of the best trained model.

**Sections:**
1. Load saved model
2. Confusion Matrix
3. Classification Report
4. ROC Curve
5. Precision-Recall Curve
6. Feature Importance
7. SHAP Analysis (if available)
8. Error Analysis

---

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)

sys.path.insert(0, os.path.join('..', 'src'))
from utils import load_dataset, load_model, plot_confusion_matrix, plot_roc_curve, plot_feature_importance, ensure_dir
from preprocessing import run_preprocessing_pipeline

try:
    import shap
    SHAP_AVAILABLE = True
    print('SHAP available ✅')
except ImportError:
    SHAP_AVAILABLE = False
    print('SHAP not installed — skipping SHAP analysis.')

sns.set_theme(style='whitegrid', font_scale=1.1)
PLOTS_DIR = os.path.join('..', 'outputs', 'plots')
ensure_dir(PLOTS_DIR)
print('Libraries loaded ✅')

## 5.1  Load Model & Recreate Test Set

In [ ]:
# Load saved model bundle
MODEL_PATH = os.path.join('..', 'model', 'churn_model.pkl')
bundle = load_model(MODEL_PATH)

model         = bundle['model']
model_name    = bundle['model_name']
scaler        = bundle['scaler']
feature_names = bundle['feature_names']

print(f'Loaded model: {model_name}')
print(f'Features    : {len(feature_names)}')

In [ ]:
# Recreate the exact same test set (same random_state=42 ensures reproducibility)
DATA_PATH = os.path.join('..', 'data', 'customer_churn.csv')
df = load_dataset(DATA_PATH)
X_train, X_test, y_train, y_test, _, _, _ = run_preprocessing_pipeline(df)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

print(f'Test set size: {len(y_test)} samples')
print(f'Churn in test: {y_test.sum()} ({y_test.mean()*100:.1f}%)')

## 5.2  Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('Confusion Matrix:')
print(f'  True Negatives  (TN): {tn}  — Correctly predicted NO churn')
print(f'  False Positives (FP): {fp}  — Predicted YES churn, actually NO')
print(f'  False Negatives (FN): {fn}  — Predicted NO churn, actually YES (missed churners)')
print(f'  True Positives  (TP): {tp}  — Correctly predicted YES churn')

plot_confusion_matrix(
    y_test, y_pred,
    save_path=os.path.join(PLOTS_DIR, 'confusion_matrix.png'),
    title=f'Confusion Matrix — {model_name}'
)

## 5.3  Classification Report

In [ ]:
print(f'Classification Report — {model_name}')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

# Save to text file
REPORTS_DIR = os.path.join('..', 'outputs', 'reports')
ensure_dir(REPORTS_DIR)
report_path = os.path.join(REPORTS_DIR, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(f'Model: {model_name}\n')
    f.write('=' * 55 + '\n')
    f.write(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
print(f'Report saved to: {report_path}')

## 5.4  ROC Curve

In [ ]:
if y_prob is not None:
    plot_roc_curve(
        model, X_test, y_test,
        model_name=model_name,
        save_path=os.path.join(PLOTS_DIR, 'roc_curve.png')
    )
    print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
else:
    print('predict_proba not available — skipping ROC curve.')

## 5.5  Precision-Recall Curve

In [ ]:
if y_prob is not None:
    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
    avg_precision = average_precision_score(y_test, y_prob)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color='purple', lw=2,
             label=f'PR Curve (AP = {avg_precision:.3f})')
    plt.axhline(y=y_test.mean(), color='red', linestyle='--',
                label=f'Baseline (prevalence = {y_test.mean():.2f})')
    plt.xlabel('Recall (Sensitivity)', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title(f'Precision-Recall Curve — {model_name}', fontsize=14, fontweight='bold')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'precision_recall_curve.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Average Precision: {avg_precision:.4f}')

## 5.6  Feature Importance

In [ ]:
importances = None
if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
    importance_type = 'Gini Importance'
elif hasattr(model, 'coef_'):
    importances = np.abs(model.coef_[0])
    importance_type = 'Coefficient Magnitude'

if importances is not None:
    plot_feature_importance(
        feature_names, importances, top_n=15,
        save_path=os.path.join(PLOTS_DIR, 'feature_importance.png'),
        title=f'Top 15 Features — {model_name} ({importance_type})'
    )

    # Print top 10 as table
    idx = np.argsort(importances)[::-1][:10]
    fi_df = pd.DataFrame({
        'Feature'   : [feature_names[i] for i in idx],
        'Importance': [round(importances[i], 5) for i in idx]
    })
    print('\nTop 10 Most Important Features:')
    print(fi_df.to_string(index=False))
else:
    print('Feature importance not available for this model type.')

## 5.7  SHAP Analysis

In [ ]:
if SHAP_AVAILABLE:
    print('Generating SHAP analysis...')
    try:
        model_type = type(model).__name__
        X_test_df  = pd.DataFrame(X_test, columns=feature_names)

        if model_type in ('RandomForestClassifier', 'DecisionTreeClassifier', 'XGBClassifier'):
            explainer  = shap.TreeExplainer(model)
            shap_vals  = explainer.shap_values(X_test_df)
            sv = shap_vals[1] if isinstance(shap_vals, list) else shap_vals
        else:
            background = shap.sample(X_test_df, 100, random_state=42)
            explainer  = shap.KernelExplainer(model.predict_proba, background)
            shap_vals  = explainer.shap_values(X_test_df.iloc[:200])
            sv = shap_vals[1] if isinstance(shap_vals, list) else shap_vals
            X_test_df  = X_test_df.iloc[:200]

        # Bar summary
        plt.figure(figsize=(10, 6))
        shap.summary_plot(sv, X_test_df, plot_type='bar', show=False)
        plt.title('SHAP Feature Importance (Mean |SHAP value|)', fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, 'shap_summary_bar.png'), dpi=150, bbox_inches='tight')
        plt.show()

        # Beeswarm
        plt.figure(figsize=(10, 8))
        shap.summary_plot(sv, X_test_df, show=False)
        plt.title('SHAP Beeswarm Plot', fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, 'shap_summary_beeswarm.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print('SHAP plots saved ✅')

    except Exception as e:
        print(f'SHAP analysis failed: {e}')
else:
    print('SHAP not available. Install with: pip install shap')

## 5.8  Error Analysis — Misclassified Samples

In [ ]:
# Identify false negatives and false positives
y_pred_arr = np.array(y_pred)
y_test_arr = np.array(y_test)

false_negatives = np.where((y_pred_arr == 0) & (y_test_arr == 1))[0]
false_positives = np.where((y_pred_arr == 1) & (y_test_arr == 0))[0]

print(f'False Negatives (missed churners)  : {len(false_negatives)}')
print(f'False Positives (false alarms)     : {len(false_positives)}')

if y_prob is not None:
    # Distribution of prediction probabilities by outcome
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    correct_mask = y_pred_arr == y_test_arr
    axes[0].hist(y_prob[correct_mask],   bins=30, alpha=0.7, color='green', label='Correct')
    axes[0].hist(y_prob[~correct_mask],  bins=30, alpha=0.7, color='red',   label='Incorrect')
    axes[0].set_title('Prediction Probability — Correct vs Incorrect', fontweight='bold')
    axes[0].set_xlabel('Predicted Churn Probability')
    axes[0].set_ylabel('Count')
    axes[0].legend()

    # Probability distribution by true class
    axes[1].hist(y_prob[y_test_arr == 0], bins=30, alpha=0.6, color='#2196F3', label='True No Churn')
    axes[1].hist(y_prob[y_test_arr == 1], bins=30, alpha=0.6, color='#F44336', label='True Churn')
    axes[1].axvline(0.5, color='black', linestyle='--', label='Decision threshold (0.5)')
    axes[1].set_title('Predicted Probability by True Class', fontweight='bold')
    axes[1].set_xlabel('Predicted Churn Probability')
    axes[1].set_ylabel('Count')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'error_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()

---
## Evaluation Summary

The model evaluation section covers:
- **Confusion Matrix** — shows TP, TN, FP, FN breakdown
- **Classification Report** — precision, recall, F1 per class
- **ROC Curve** — overall discriminative ability (AUC)
- **Precision-Recall Curve** — performance on the minority (Churn) class
- **Feature Importance** — which variables most influence predictions
- **SHAP** — direction and magnitude of each feature's contribution
- **Error Analysis** — understanding where the model makes mistakes

➡  **Next:** `06_prediction_demo.ipynb` — interactive single-customer prediction demo.